# Delta Lake Basics — 01: Your First Delta Table

## What you will learn
Delta Lake adds ACID transactions, versioning, and schema enforcement
to files on your storage. Understanding what Delta IS (and isn't)
is the foundation for everything else.

In this notebook:
1. What Delta Lake is and how it differs from plain Parquet
2. Creating a Delta table — three different ways
3. The `_delta_log` directory — what's inside and why it matters
4. Reading a Delta table
5. Table metadata — DESCRIBE DETAIL
6. Delta vs Parquet — side by side comparison


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/delta_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('delta-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Delta Lake ready")

## Step 1 — What is Delta Lake?

Delta Lake is NOT a new file format. It is a **storage layer** on top of Parquet.

```
Delta Table = Parquet files + _delta_log/

your_table/
├── _delta_log/                    ← Delta's transaction log
│   ├── 00000000000000000000.json  ← commit 0: CREATE TABLE
│   ├── 00000000000000000001.json  ← commit 1: INSERT
│   └── 00000000000000000010.checkpoint.parquet  ← checkpoint
└── part-00001-abc123.parquet      ← actual data (standard Parquet)
    part-00002-def456.parquet
    ...
```

Every write creates a new JSON commit file. This gives you:
- **ACID transactions** — readers always see a consistent state
- **Time travel** — query any previous version
- **Schema enforcement** — bad writes are rejected
- **Audit log** — complete history of every operation


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/delta_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('delta-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Delta Lake ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

## Step 2 — Creating a Delta Table: Three Ways


In [ ]:
TABLE_PATH = f"{DATA_DIR}/01_first_table"
shutil.rmtree(TABLE_PATH, ignore_errors=True)

# Way 1: DataFrame write API (most common)
print("Way 1: DataFrame write API")
df.write.format("delta").mode("overwrite").save(TABLE_PATH)
print(f"  Rows: {spark.read.format('delta').load(TABLE_PATH).count():,}")
print(f"  Path: {TABLE_PATH}")

In [ ]:
# Way 2: SQL CREATE TABLE (path-based)
SQL_PATH = f"{DATA_DIR}/01_sql_table"
shutil.rmtree(SQL_PATH, ignore_errors=True)

print("Way 2: SQL CREATE TABLE (path-based)")
spark.sql(f"""
    CREATE TABLE delta.`{SQL_PATH}` (
        order_id    BIGINT,
        customer_id BIGINT,
        product     STRING,
        category    STRING,
        country     STRING,
        quantity    INT,
        price       DOUBLE,
        revenue     DOUBLE,
        status      STRING,
        order_date  DATE
    ) USING delta
""")
df.write.format("delta").mode("append").save(SQL_PATH)
print(f"  Rows: {spark.read.format('delta').load(SQL_PATH).count():,}")

In [ ]:
# Way 3: DeltaTable.createIfNotExists (programmatic)
print("Way 3: DeltaTable.createIfNotExists")
API_PATH = f"{DATA_DIR}/01_api_table"
shutil.rmtree(API_PATH, ignore_errors=True)

DeltaTable.createIfNotExists(spark) \
    .location(API_PATH) \
    .addColumn("order_id",    "BIGINT") \
    .addColumn("customer_id", "BIGINT") \
    .addColumn("product",     "STRING") \
    .addColumn("category",    "STRING") \
    .addColumn("revenue",     "DOUBLE") \
    .addColumn("order_date",  "DATE") \
    .execute()

df.select("order_id","customer_id","product","category","revenue","order_date") \
  .write.format("delta").mode("append").save(API_PATH)
print(f"  Rows: {spark.read.format('delta').load(API_PATH).count():,}")

## Step 3 — Inside _delta_log: The Transaction Log

Every operation writes a JSON commit to `_delta_log/`.
Let's read it to understand what Delta records.


In [ ]:
import json as pyjson

log_path = pathlib.Path(f"{TABLE_PATH}/_delta_log")
commits  = sorted(log_path.glob("*.json"))

print(f"_delta_log contents: {len(commits)} commit file(s)")
print()

for commit_file in commits:
    print(f"=== {commit_file.name} ===")
    with open(commit_file) as f:
        for line in f:
            entry = pyjson.loads(line.strip())
            key   = list(entry.keys())[0]
            if key == "commitInfo":
                print(f"  commitInfo: operation={entry[key].get('operation')}")
            elif key == "metaData":
                schema_str = entry[key].get("schemaString","")
                cols = [f["name"] for f in pyjson.loads(schema_str).get("fields",[])]
                print(f"  metaData  : schema columns={cols}")
            elif key == "add":
                size_kb = entry[key].get("size",0)//1024
                records = pyjson.loads(entry[key].get("stats","{}")).get("numRecords","?")
                print(f"  add file  : {entry[key]['path'][-40:]} ({size_kb} KB, {records} rows)")
            elif key == "protocol":
                print(f"  protocol  : minReaderVersion={entry[key].get('minReaderVersion')}")

## Step 4 — Reading a Delta Table


In [ ]:
# Standard read
df_read = spark.read.format("delta").load(TABLE_PATH)
print("Standard read:")
df_read.printSchema()
df_read.show(5)
print(f"Total rows: {df_read.count():,}")

In [ ]:
# DESCRIBE DETAIL — rich table metadata
print("DESCRIBE DETAIL:")
DeltaTable.forPath(spark, TABLE_PATH).detail() \
    .select("format","numFiles","sizeInBytes","partitionColumns","properties") \
    .show(truncate=False)

# Table history
print("\nDESCRIBE HISTORY:")
DeltaTable.forPath(spark, TABLE_PATH).history() \
    .select("version","timestamp","operation","operationMetrics") \
    .show(truncate=False)

## Step 5 — Delta vs Parquet: Side by Side


In [ ]:
PQ_PATH = f"{DATA_DIR}/01_plain_parquet"
df.write.mode("overwrite").parquet(PQ_PATH)

# Size comparison
delta_files = list(pathlib.Path(TABLE_PATH).rglob("*.parquet"))
delta_mb    = sum(f.stat().st_size for f in delta_files) / 1024/1024

pq_files = list(pathlib.Path(PQ_PATH).rglob("*.parquet"))
pq_mb    = sum(f.stat().st_size for f in pq_files) / 1024/1024

log_files = list(pathlib.Path(f"{TABLE_PATH}/_delta_log").rglob("*"))
log_kb    = sum(f.stat().st_size for f in log_files if f.is_file()) / 1024

print("Delta vs Parquet comparison:")
print(f"  Plain Parquet  : {len(pq_files)} file(s), {pq_mb:.1f} MB")
print(f"  Delta Parquet  : {len(delta_files)} file(s), {delta_mb:.1f} MB  (same data)")
print(f"  Delta _delta_log: {len(log_files)} file(s), {log_kb:.1f} KB overhead")
print()
print("Delta overhead: tiny JSON log files (~KB) on top of standard Parquet")
print("The Parquet files are 100% identical — Delta just adds the log layer")
print()
print("What Delta adds:")
print("  ✅ ACID transactions — concurrent write safety")
print("  ✅ Time travel      — query any version")
print("  ✅ Schema evolution — managed column changes")
print("  ✅ Audit log        — who did what and when")
print("  ❌ More metadata    — tiny overhead in _delta_log")

## Summary

### Three ways to create a Delta table
```python
# 1. Write API
df.write.format("delta").mode("overwrite").save(path)

# 2. SQL
spark.sql(f"CREATE TABLE delta.`{path}` (...) USING delta")

# 3. DeltaTable builder
DeltaTable.createIfNotExists(spark).location(path).addColumn(...).execute()
```

### Key concepts
- Delta = Parquet files + `_delta_log/` JSON commits
- Every write = new commit file in `_delta_log/`
- Readers always see the latest committed state
- `_delta_log` is tiny (KB) — not a performance concern

**Next:** `02_reading_writing.ipynb` — all read/write options and modes.
